# 06 · Predicting Segments for New Customers

We load the trained Random Forest model, predict the segment of ten hypothetical new customers, and check that the predictions are consistent with the value ranges of each segment.

## 1. Load model and data

In [1]:
import joblib
import pandas as pd

MODEL_PATH = "../models/segmentation_model.pkl"
SEGMENTED_PARQUET = "../data/processed/segmented_customers.parquet"

model = joblib.load(MODEL_PATH)
DATA = pd.read_parquet(SEGMENTED_PARQUET)

## 2. Predict new customers

In [2]:
new_customers = pd.DataFrame({
    'conversions':   [5,  45, 20, 38, 2,  40, 15, 35, 8,  42],
    'impressions':   [30000, 10000, 22000, 40000, 25000, 12000, 18000, 38000, 28000, 9000],
    'clicks':        [100, 400, 800, 500, 150, 350, 900, 450, 200, 380],
    'monthly_spend': [500, 3500, 1200, 2800, 800, 3200, 1500, 2600, 600, 3800]
})

predictions = model.predict(new_customers)

segment_names = {
    0: 'Ghosts',
    1: 'Whales',
    2: 'Window Shoppers',
    3: 'Core Buyers'
}

new_customers['segment'] = predictions
new_customers['segment_name'] = new_customers['segment'].map(segment_names)
print(new_customers)

   conversions  impressions  clicks  monthly_spend  segment     segment_name
0            5        30000     100            500        0           Ghosts
1           45        10000     400           3500        1           Whales
2           20        22000     800           1200        2  Window Shoppers
3           38        40000     500           2800        3      Core Buyers
4            2        25000     150            800        0           Ghosts
5           40        12000     350           3200        1           Whales
6           15        18000     900           1500        2  Window Shoppers
7           35        38000     450           2600        3      Core Buyers
8            8        28000     200            600        0           Ghosts
9           42         9000     380           3800        1           Whales


## 3. Sanity check

Do the predictions make sense given the value ranges of each feature per segment?

In [3]:
DATA.groupby('segment_name')[['conversions', 'impressions', 'clicks', 'monthly_spend']].describe().round(0)

conversions                                            \
                      count  mean   std   min   25%   50%   75%   max   
segment_name                                                            
Core Buyers           121.0  38.0   9.0  18.0  30.0  40.0  46.0  49.0   
Ghosts                117.0  11.0   8.0   0.0   5.0   9.0  16.0  32.0   
Whales                129.0  34.0  11.0   4.0  28.0  36.0  42.0  49.0   
Window Shoppers       132.0  18.0  12.0   0.0   9.0  18.0  25.0  48.0   

                impressions           ... clicks        monthly_spend          \
                      count     mean  ...    75%    max         count    mean   
segment_name                          ...                                       
Core Buyers           121.0  38416.0  ...  677.0  991.0         121.0  2651.0   
Ghosts                117.0  28886.0  ...  377.0  619.0         117.0  2289.0   
Whales                129.0  12240.0  ...  577.0  984.0         129.0  3350.0   
Window Shoppers       132.0  20911.0  ...  898.0  995.0         132.0  1909.0   

                                                                
                    std    min     25%     50%     75%     max  
segment_name                                                    
Core Buyers      1100.0  106.0  2568.0  2572.0  3294.0  4953.0  
Ghosts           1079.0  160.0  1638.0  2572.0  2572.0  4934.0  
Whales           1060.0  862.0  2572.0  3218.0  4279.0  4996.0  
Window Shoppers  1019.0  108.0   966.0  2173.0  2572.0  4444.0  

[4 rows x 32 columns]